In [1]:
import pandas as pd
import pyodbc
from config import server, database, username, password
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import os
from datetime import datetime
from reportlab.lib import colors
from reportlab.lib.pagesizes import letter, landscape
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph, Spacer, PageBreak, Image
from reportlab.lib.styles import getSampleStyleSheet
from urllib.parse import quote

def agregar_seccion_notas_cafe(elements, df_notas, sap_value, tienda_value, style_h2, style_h3, pre_space_pts=24):
    import pandas as pd
    notas = df_notas.copy()
    sap_str = str(sap_value)
    if 'sap' in notas.columns:
        notas['sap'] = notas['sap'].astype(str)
        notas = notas[notas['sap'] == sap_str]
    elif 'Sap' in notas.columns:
        notas['Sap'] = notas['Sap'].astype(str)
        notas = notas[notas['Sap'] == sap_str]
    def col_like(name):
        for c in notas.columns:
            if c.lower() == name.lower():
                return c
        return None
    c_fecha = col_like('fecha')
    c_folio = col_like('folio')
    c_coment = col_like('Comentario')
    c_importe = col_like('importe')
    h2_text = "El ultimo listado disponible de notas pendientes de captura tiene estos registros:"
    elements.append(Spacer(1, pre_space_pts))
    elements.append(Paragraph(h2_text, style_h2))
    if not all([c_fecha, c_folio, c_coment, c_importe]):
        elements.append(Spacer(1, 8))
        elements.append(Paragraph("No se pudo construir la tabla de notas: faltan columnas esperadas (fecha, folio, Comentario, importe).", style_h3))
        elements.append(Spacer(1, 12))
        return
    if notas.empty:
        elements.append(Paragraph("<i>No hay notas pendientes de captura para este SAP.</i>", style_h3))
        elements.append(Spacer(1, 12))
        return
    if pd.api.types.is_datetime64_any_dtype(notas[c_fecha]):
        notas[c_fecha] = notas[c_fecha].dt.strftime('%d/%m/%Y')
    else:
        notas[c_fecha] = notas[c_fecha].astype(str)
    notas[c_importe] = pd.to_numeric(notas[c_importe], errors='coerce').fillna(0).astype(int)
    def fmt_miles(n):
        return f"{int(n):,}"
    data = [['Fecha', 'Folio', 'Comentario', 'Importe']]
    for _, r in notas.iterrows():
        data.append([r[c_fecha], str(r[c_folio]), str(r[c_coment]), fmt_miles(r[c_importe])])
    total = int(notas[c_importe].sum())
    data.append(['', '', 'Total', fmt_miles(total)])
    cafe_oscuro = colors.HexColor("#7B5E3B")
    cafe_claro = colors.HexColor("#F3E9DD")
    borde_cafe = colors.HexColor("#C8B299")
    total_bg = colors.HexColor("#E8D9C4")
    table_cafe = Table(data, repeatRows=1)
    style_cafe = TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), cafe_oscuro),
        ('TEXTCOLOR',  (0, 0), (-1, 0), colors.white),
        ('FONTNAME',   (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('ALIGN',      (0, 0), (-1, 0), 'CENTER'),
        ('ROWBACKGROUNDS', (0, 1), (-1, -2), [cafe_claro, colors.white]),
        ('BACKGROUND', (0, -1), (-1, -1), total_bg),
        ('FONTNAME',   (0, -1), (-1, -1), 'Helvetica-Bold'),
        ('ALIGN',      (-1, 1), (-1, -1), 'RIGHT'),
        ('VALIGN',     (0, 0), (-1, -1), 'MIDDLE'),
        ('GRID',       (0, 0), (-1, -1), 0.5, borde_cafe),
        ('LEFTPADDING',  (0,0), (-1,-1), 6),
        ('RIGHTPADDING', (0,0), (-1,-1), 6),
        ('TOPPADDING',   (0,0), (-1,-1), 3),
        ('BOTTOMPADDING',(0,0), (-1,-1), 3),
    ])
    table_cafe.setStyle(style_cafe)
    elements.append(table_cafe)
    elements.append(Spacer(1, 12))

connection_string = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"UID={username};"
    f"PWD={password}"
)
conn = pyodbc.connect(connection_string)

with open('my_query3.sql', 'r', encoding='utf-8') as file:
    query = file.read()
with open('my_query4.sql', 'r', encoding='utf-8') as file:
    query2 = file.read()
with open('my_query6.sql', 'r') as file:
    query3 = file.read()
df_notas = pd.read_sql(query3, conn)
with open('my_query7.sql', 'r') as file:
    query4 = file.read()
df_cabeceras = pd.read_sql(query4, conn)
with open('my_query8.sql', 'r') as file:
    query5 = file.read()
df_trade = pd.read_sql(query5, conn)
df_tops_tienda = pd.read_sql(query, conn)
df_dir_whatsapp = pd.read_sql(query2, conn)
df_alertas = pd.read_excel('Resumen_carga.xlsx', sheet_name='resumen_pedido', header=4, engine='openpyxl')
df_alertas = df_alertas.sort_values(by=['Sap', 'Familia', 'linea', 'Presentacion', 'Sku'])
df_alertas = df_alertas[(df_alertas['Activa_cliente'] == 1) | (df_alertas['Puede_pedir_op'] == 1)]
df_alertas = df_alertas[df_alertas['Con_algun_empuje'] == 1]
df_alertas = df_alertas.merge(
    df_tops_tienda.rename(columns={'sap': 'Sap', 'sku': 'Sku'})[['Sap', 'Sku', 'top_tienda', 'Menor3']],
    on=['Sap', 'Sku'],
    how='left'
)
df_alertas['Piezas adicionales'] = df_alertas['Piezas a cargar'] - df_alertas['Pedido_original_cadena']
df_alertas = df_alertas[~((df_alertas['Grupo'] == 'Wal-Mart') & (df_alertas['Presentacion'] == 'GRANEL'))]
df_alertas = df_alertas.merge(df_dir_whatsapp[['Sap', 'Cel_ejecutivo', 'Cel_coordinadora']], on='Sap', how='left')
df_alertas1 = pd.read_excel('Resumen_carga.xlsx', sheet_name='resumen_pedido', header=4, engine='openpyxl')
df_alertas1 = df_alertas1.sort_values(by=['Sap', 'Familia', 'linea', 'Presentacion', 'Sku'])
df_alertas1 = df_alertas1[df_alertas1['Con_algun_recorte'] == 1]
df_alertas1.to_csv('test.csv', index=False)

output_folder = os.path.join('alertas_wp', 'impulso')
os.makedirs(output_folder, exist_ok=True)
output_folder1 = os.path.join('alertas_wp', 'pedido_soriana')
os.makedirs(output_folder1, exist_ok=True)
for filename in os.listdir(output_folder):
    file_path = os.path.join(output_folder, filename)
    if os.path.isfile(file_path):
        os.remove(file_path)
for filename in os.listdir(output_folder1):
    file_path1 = os.path.join(output_folder1, filename)
    if os.path.isfile(file_path1):
        os.remove(file_path1)

fecha_actual = datetime.now()
fecha_nombre = fecha_actual.strftime('%y%m%d')
meses_es = ['enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio', 'julio', 'agosto', 'septiembre', 'octubre', 'noviembre', 'diciembre']
fecha_h2 = f"{fecha_actual.day} {meses_es[fecha_actual.month - 1]} {fecha_actual.year}"

styles = getSampleStyleSheet()
style_h1 = styles['Heading1']
style_h2 = styles['Heading2']
style_h3 = styles['Heading3']

columnas_azul = ['Sku', 'Producto', 'Scan_prom', 'Pedido_original_cadena', 'Piezas adicionales']
columnas_roja = ['Sku', 'Producto', 'Merma_%']

BLUE = colors.HexColor("#007BFF")
LIGHT_BLUE = colors.lightblue
RED = colors.HexColor("#DC3545")
LIGHT_RED = colors.HexColor("#F8D7DA")
DARK_GREEN = colors.HexColor("#28A745")
LIGHT_GREEN = colors.lightgreen

rows_csv = []
base_url = 'https://soporteracu.com/shared/analytics/impulso_pdf/'
rows_csv_soriana = []
base_url_soriana = 'https://soporteracu.com/shared/analytics/pedido_soriana/'
logo_path = 'loguito.png'

def formatear_fecha_es(valor):
    try:
        dt = pd.to_datetime(valor, errors='coerce')
        if pd.isna(dt):
            return str(valor)
        return f"{dt.day} {meses_es[dt.month - 1]} {dt.year}"
    except:
        return str(valor)

for sap_value, group_df in df_alertas.groupby('Sap'):
    tienda_value = group_df['Tienda'].iloc[0]
    data_azul = [columnas_azul]
    familia_anterior = None
    for _, row in group_df.iterrows():
        if row['Familia'] != familia_anterior:
            data_azul.append([f"Familia: {row['Familia']}", '', '', '', ''])
            familia_anterior = row['Familia']
        fila = [str(row['Sku']), str(row['Producto']), str(row['Scan_prom']), str(row['Pedido_original_cadena']), f"{row['Piezas adicionales']}"]
        data_azul.append(fila)
    pdf_filename = f"{fecha_nombre}_{sap_value}_{tienda_value}_oportunidad.pdf"
    pdf_path = os.path.join(output_folder, pdf_filename)
    doc = SimpleDocTemplate(pdf_path, pagesize=landscape(letter), topMargin=20, bottomMargin=20, leftMargin=36, rightMargin=36)
    elements = []
    if os.path.exists(logo_path):
        logo = Image(logo_path)
        logo.drawHeight = 50
        logo.drawWidth = 70
        logo.hAlign = 'LEFT'
        elements.append(logo)
        elements.append(Spacer(1, 12))
    elements.append(Paragraph(f"SAP: {sap_value} | Tienda: {tienda_value}", style_h1))
    elements.append(Paragraph(f"Pedidos del {fecha_h2}", style_h2))
    elements.append(Paragraph("Sigma Analytics", style_h3))
    elements.append(Spacer(1, 12))
    table_azul = Table(data_azul, repeatRows=1)
    style_azul = TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), BLUE),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.whitesmoke, LIGHT_BLUE]),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
    ])
    table_azul.setStyle(style_azul)
    elements.append(table_azul)
    df_red = df_alertas1[df_alertas1['Sap'] == sap_value].copy()
    if not df_red.empty:
        elements.append(PageBreak())
        tienda_value_red = (df_red['Tienda'].dropna().astype(str).iloc[0] if 'Tienda' in df_red.columns and not df_red['Tienda'].dropna().empty else str(tienda_value))
        if os.path.exists(logo_path):
            logo2 = Image(logo_path)
            logo2.drawHeight = 50
            logo2.drawWidth = 70
            logo2.hAlign = 'LEFT'
            elements.append(logo2)
            elements.append(Spacer(1, 12))
        elements.append(Paragraph(f"SAP: {sap_value} - {tienda_value_red}", style_h1))
        elements.append(Paragraph("Estos productos han tenido merma alta, sugerimos no hacer pedidos adicionales, lo necesario será gestionado por el area de resurtido. En caso de dudas por favor comunicate con tu ejecutivo.", style_h2))
        elements.append(Paragraph("Sigma Analytics", style_h3))
        elements.append(Spacer(1, 12))
        data_roja = [columnas_roja]
        fam_ant = None
        df_red_sorted = df_red.sort_values(by=['Familia', 'linea', 'Presentacion', 'Sku'])
        for _, r in df_red_sorted.iterrows():
            if r['Familia'] != fam_ant:
                data_roja.append([f"Familia: {r['Familia']}", '', ''])
                fam_ant = r['Familia']
            vol = r.get('Vol_promedio', 0)
            merma = r.get('Merma_promedio', 0)
            if pd.isna(vol) or vol == 0:
                ratio_str = "1000%"
            else:
                ratio_val = (-merma / vol)
                if ratio_val < 0:
                    ratio_str = "1000%"
                else:
                    ratio_str = f"{ratio_val * 100:.1f}%"
            data_roja.append([str(r['Sku']), str(r['Producto']), ratio_str])
        table_roja = Table(data_roja, repeatRows=1)
        style_roja = TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), RED),
            ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
            ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
            ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.whitesmoke, LIGHT_RED]),
            ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
        ])
        table_roja.setStyle(style_roja)
        elements.append(table_roja)
        agregar_seccion_notas_cafe(elements=elements, df_notas=df_notas, sap_value=sap_value, tienda_value=tienda_value_red, style_h2=style_h2, style_h3=style_h3, pre_space_pts=24)
    else:
        agregar_seccion_notas_cafe(elements=elements, df_notas=df_notas, sap_value=sap_value, tienda_value=tienda_value, style_h2=style_h2, style_h3=style_h3, pre_space_pts=24)

    elements.append(PageBreak())
    if os.path.exists(logo_path):
        logo3 = Image(logo_path)
        logo3.drawHeight = 50
        logo3.drawWidth = 70
        logo3.hAlign = 'LEFT'
        elements.append(logo3)
        elements.append(Spacer(1, 12))
    elements.append(Paragraph(f"SAP: {sap_value} | Tienda: {tienda_value}", style_h1))
    elements.append(Paragraph(f"Pedidos del {fecha_h2}", style_h2))
    elements.append(Paragraph("Sigma Analytics", style_h3))
    elements.append(Spacer(1, 12))
    elements.append(Paragraph("Informacion de cabeceras activas de esta tienda", style_h2))
    elements.append(Spacer(1, 8))
    df_cab_sap = df_cabeceras.copy()
    if 'Sap' in df_cab_sap.columns:
        df_cab_sap = df_cab_sap[df_cab_sap['Sap'] == sap_value]
    elif 'sap' in df_cab_sap.columns:
        df_cab_sap['sap'] = df_cab_sap['sap'].astype(type(sap_value))
        df_cab_sap = df_cab_sap[df_cab_sap['sap'] == sap_value]
    cols_cab = []
    for c in df_cab_sap.columns:
        cols_cab.append(c)
    if not df_cab_sap.empty:
        df_cab_sap = df_cab_sap.copy()
        if 'Inicio_vigencia' in df_cab_sap.columns:
            df_cab_sap['Inicio_vigencia_fmt'] = df_cab_sap['Inicio_vigencia'].apply(formatear_fecha_es)
        else:
            df_cab_sap['Inicio_vigencia_fmt'] = ''
        if 'Fin_vigencia' in df_cab_sap.columns:
            df_cab_sap['Fin_vigencia_fmt'] = df_cab_sap['Fin_vigencia'].apply(formatear_fecha_es)
        else:
            df_cab_sap['Fin_vigencia_fmt'] = ''
        oh = pd.to_numeric(df_cab_sap.get('OH_kilos', pd.Series([None]*len(df_cab_sap))), errors='coerce')
        peso = pd.to_numeric(df_cab_sap.get('Peso', pd.Series([None]*len(df_cab_sap))), errors='coerce')
        inv_teorico = (oh / peso).replace([pd.NA, pd.NaT, float('inf'), -float('inf')], pd.NA)
        inv_teorico = inv_teorico.fillna(0).round(0).astype(int)
        df_cab_sap['Inventario teorico'] = inv_teorico
        header_cab = ['Sku', 'Producto', 'Inicio vigencia', 'Fin vigencia', 'Tipo merma', 'Objetivo piezas', 'Inventario teorico']
        data_cab = [header_cab]
        for _, r in df_cab_sap.iterrows():
            sku = r.get('Sku', '')
            prod = r.get('Prooducto', '')
            ini = r.get('Inicio_vigencia_fmt', '')
            fin = r.get('Fin_vigencia_fmt', '')
            tipo_merma = r.get('Tipo_merma_ant', '')
            objetivo = r.get('Objetivo_OH', '')
            inv = r.get('Inventario teorico', 0)
            data_cab.append([str(sku), str(prod), str(ini), str(fin), str(tipo_merma), str(objetivo), str(int(inv))])
        table_cab = Table(data_cab, repeatRows=1)
        style_cab = TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), DARK_GREEN),
            ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
            ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
            ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.whitesmoke, LIGHT_GREEN]),
            ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
        ])
        table_cab.setStyle(style_cab)
        elements.append(table_cab)
    else:
        elements.append(Paragraph("<i>No hay cabeceras activas para este SAP.</i>", style_h3))
    elements.append(Spacer(1, 18))
    elements.append(Paragraph("Activaciones en la tienda:", style_h2))
    elements.append(Spacer(1, 8))
    df_trade_sap = df_trade.copy()
    if 'Sap' in df_trade_sap.columns:
        df_trade_sap = df_trade_sap[df_trade_sap['Sap'] == sap_value]
    elif 'sap' in df_trade_sap.columns:
        df_trade_sap['sap'] = df_trade_sap['sap'].astype(type(sap_value))
        df_trade_sap = df_trade_sap[df_trade_sap['sap'] == sap_value]
    if not df_trade_sap.empty:
        df_trade_sap = df_trade_sap.copy()
        if 'Inicio_vigencia' in df_trade_sap.columns:
            df_trade_sap['Inicio_vigencia_fmt'] = df_trade_sap['Inicio_vigencia'].apply(formatear_fecha_es)
        else:
            df_trade_sap['Inicio_vigencia_fmt'] = ''
        if 'Fin_vigencia' in df_trade_sap.columns:
            df_trade_sap['Fin_vigencia_fmt'] = df_trade_sap['Fin_vigencia'].apply(formatear_fecha_es)
        else:
            df_trade_sap['Fin_vigencia_fmt'] = ''
        def transformar_desc_trade(x):
            if pd.isna(x):
                return ''
            s = str(x).strip()
            val = pd.to_numeric(s, errors='coerce')
            if pd.isna(val):
                return s
            return f"${val:,.0f} precio sugerido"
        df_trade_sap['Clase_fmt'] = df_trade_sap.get('Desc_trade', '').apply(transformar_desc_trade)
        header_trade = ['Sku', 'Producto', 'Clase', 'Inicio vigencia', 'Fin vigencia']
        data_trade = [header_trade]
        for _, r in df_trade_sap.iterrows():
            sku = r.get('Sku', '')
            prod = r.get('Producto', '')
            clase = r.get('Clase_fmt', '')
            ini = r.get('Inicio_vigencia_fmt', '')
            fin = r.get('Fin_vigencia_fmt', '')
            data_trade.append([str(sku), str(prod), str(clase), str(ini), str(fin)])
        table_trade = Table(data_trade, repeatRows=1)
        style_trade = TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), DARK_GREEN),
            ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
            ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
            ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
            ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.whitesmoke, LIGHT_GREEN]),
            ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
        ])
        table_trade.setStyle(style_trade)
        elements.append(table_trade)
    else:
        elements.append(Paragraph("<i>No hay activaciones registradas para esta tienda.</i>", style_h3))

    doc.build(elements)
    tienda_encoded = quote(str(tienda_value), safe='()')
    url_filename = f"{fecha_nombre}_{sap_value}_{tienda_encoded}_oportunidad.pdf"
    url_full = base_url + url_filename
    if group_df['Grupo'].iloc[0] != 'Soriana':
        cel_ejec_series = group_df['Cel_ejecutivo'].dropna().astype(str)
        cel_coord_series = group_df['Cel_coordinadora'].dropna().astype(str)
        cel_ejec = cel_ejec_series.iloc[0].strip() if not cel_ejec_series.empty else ''
        cel_coord = cel_coord_series.iloc[0].strip() if not cel_coord_series.empty else ''
        rows_csv.append({'Telefono': cel_ejec, 'Url': url_full})
        if cel_coord and cel_coord != '0':
            rows_csv.append({'Telefono': cel_coord, 'Url': url_full})

csv_path = os.path.join(os.getcwd(), 'archivo.csv')
pd.DataFrame(rows_csv, columns=['Telefono', 'Url']).to_csv(csv_path, index=False, encoding='utf-8-sig')

df_alertas_soriana = df_alertas[df_alertas['Grupo'] == 'Soriana'].copy()
columnas_soriana = ['Sku', 'Producto', 'Scan_prom', 'Piezas a cargar']

for sap_value, group_df in df_alertas_soriana.groupby('Sap'):
    tienda_value = group_df['Tienda'].iloc[0]
    familia_anterior = None
    data = [columnas_soriana]
    for _, row in group_df.iterrows():
        if row['Familia'] != familia_anterior:
            data.append([f"Familia: {row['Familia']}", '', '', ''])
            familia_anterior = row['Familia']
        fila = [str(row['Sku']), str(row['Producto']), str(row['Scan_prom']), str(row['Piezas a cargar'])]
        data.append(fila)
    pdf_filename = f"{fecha_nombre}_{sap_value}_{tienda_value}_pedido.pdf"
    pdf_path = os.path.join(output_folder1, pdf_filename)
    doc = SimpleDocTemplate(pdf_path, pagesize=landscape(letter), topMargin=20, bottomMargin=20, leftMargin=36, rightMargin=36)
    elements = []
    if os.path.exists(logo_path):
        logo = Image(logo_path)
        logo.drawHeight = 50
        logo.drawWidth = 70
        logo.hAlign = 'LEFT'
        elements.append(logo)
        elements.append(Spacer(1, 12))
    elements.append(Paragraph(f"SAP: {sap_value} | Tienda: {tienda_value}", style_h1))
    elements.append(Paragraph(f"Pedidos del {fecha_h2}", style_h2))
    elements.append(Paragraph("Sigma Analytics", style_h3))
    elements.append(Spacer(1, 12))
    table = Table(data, repeatRows=1)
    style = TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor("#28A745")),
        ('TEXTCOLOR', (0, 0), (-1, 0), colors.white),
        ('FONTNAME', (0, 0), (-1, 0), 'Helvetica-Bold'),
        ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
        ('ROWBACKGROUNDS', (0, 1), (-1, -1), [colors.whitesmoke, colors.lightgreen]),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.grey),
    ])
    table.setStyle(style)
    elements.append(table)
    doc.build(elements)
    tienda_encoded = quote(str(tienda_value), safe='()')
    url_filename = f"{fecha_nombre}_{sap_value}_{tienda_encoded}_pedido.pdf"
    url_full = base_url_soriana + url_filename
    cel_ejec_series = group_df['Cel_ejecutivo'].dropna().astype(str)
    cel_coord_series = group_df['Cel_coordinadora'].dropna().astype(str)
    cel_ejec = cel_ejec_series.iloc[0].strip() if not cel_ejec_series.empty else ''
    cel_coord = cel_coord_series.iloc[0].strip() if not cel_coord_series.empty else ''
    rows_csv_soriana.append({'Telefono': cel_ejec, 'Url': url_full})
    if cel_coord and cel_coord != '0':
        rows_csv_soriana.append({'Telefono': cel_coord, 'Url': url_full})

print("✅ PDFs generados exitosamente en la carpeta 'alertas_wp/impulso'.")
print(f"🧾 CSV generado: {csv_path}")
print("✅ PDFs para Soriana generados exitosamente en la carpeta 'alertas_wp/pedido_soriana'.")


✅ PDFs generados exitosamente en la carpeta 'alertas_wp/impulso'.
🧾 CSV generado: c:\Users\sigma.analytics\OneDrive - Sigma\Morales Mercado, Enrique Guadalupe's files - Celula_proyeccion\Python\RL_Chedraui_y_ajustes_total\archivo.csv
✅ PDFs para Soriana generados exitosamente en la carpeta 'alertas_wp/pedido_soriana'.


In [2]:
df_alertas.head()

,Fecha_Invetario,Sap,Tienda,Grupo,cadena,Cedi,Region,Sku,Producto,Familia,...,Unnamed: 66,Unnamed: 67,Unnamed: 68,Unnamed: 69,Unnamed: 70,top_tienda,Menor3,Piezas adicionales,Cel_ejecutivo,Cel_coordinadora
0,2026-01-17,310963,BAU (3767 PACHUCA),Wal-Mart,BODEGA AURRERA,Pachuca,Centro,619,CASERO CHORIZO DE CERDO 200GR CHX,CARNES FRIAS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5,7712208884,8120229929
1,2026-01-17,310963,BAU (3767 PACHUCA),Wal-Mart,BODEGA AURRERA,Pachuca,Centro,2704,CHORIZO PARA ASAR ORIGINAL 400 GRS CHX,CARNES FRIAS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,19,7712208884,8120229929
2,2026-01-17,310963,BAU (3767 PACHUCA),Wal-Mart,BODEGA AURRERA,Pachuca,Centro,2364,JAMON AMERICANO DE PAVO FUD 196 G,CARNES FRIAS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,7712208884,8120229929
3,2026-01-17,310963,BAU (3767 PACHUCA),Wal-Mart,BODEGA AURRERA,Pachuca,Centro,498,COCIDO REB CHIMEX 900 GRS,CARNES FRIAS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,7712208884,8120229929
4,2026-01-17,310963,BAU (3767 PACHUCA),Wal-Mart,BODEGA AURRERA,Pachuca,Centro,2349,PECHUGA DE PAVO REB SANDWICH 200 G SRF,CARNES FRIAS,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9,7712208884,8120229929
